Primeiro passo é uma extração simples de dados obter todos os codigos das ações.
Vou retirar através de scraping desse site aqui https://www.dadosdemercado.com.br/acoes

In [ ]:
pip install pyspark beautifulsoup4 requests yfinance pyarrow

In [ ]:
import requests
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataFrameCreation").getOrCreate()

def extract_stock_data():
    url = "https://www.dadosdemercado.com.br/acoes"

    # Use a user-agent to mimic a real browser request
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        # Fetch the webpage content
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Check for connection errors

        # Parse the HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the stock data table
        table = soup.find('table')

        if not table:
            print("Table not found on the page.")
            return

        data = []

        # Iterate over all rows in the table
        # We search for 'tr' tags which represent table rows
        rows = table.find_all('tr')

        for row in rows:
            # Find all cells ('td') in the row
            cols = row.find_all('td')

            # We need at least 2 columns (Ticker is usually col 0, Nome is col 1)
            # This check also skips the header row if it uses 'th' tags
            if len(cols) >= 2:
                ticker = cols[0].get_text(strip=True)
                nome = cols[1].get_text(strip=True)

                # Skip if we accidentally picked up the header row text
                if ticker == "Ticker":
                    continue

                # Add to our list with 'Nome' renamed to 'name'
                data.append({
                    "Ticker": ticker,
                    "Name": nome
                })

        # Create a DataFrame for clean display or export
        df = spark.createDataFrame(data)

        # Display the first few rows
        print(f"Extracted {df.count()} stocks.")
        print(df.head())

        # Optional: Save to CSV
        # df.to_csv('stock_data.csv', index=False)

        return df

    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
df = extract_stock_data()

Extracted 394 stocks.
Row(Name='B3', Ticker='B3SA3')


Agora que eu tenho os dados de código e nome vou usar o apifinance do yahoo para pegar os dados de preço.

In [ ]:
import sys
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType, DateType
from datetime import datetime

# Standard imports
import yfinance as yf

# 2. Define parameters for the history fetch
# Note: 60m interval is valid for max 730 days back.
START_DATE = "2025-12-12"
END_DATE = "2025-12-17"

# 3. Define the output schema (Includes Datetime now)
result_schema2 = StructType([
    StructField("Ticker", StringType(), True),
    StructField("name", StringType(), True),
    StructField("Date", DateType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", LongType(), True)
])

# 4. Worker Function (Runs on Executor Nodes)
def process_partition_hourly(iterator):
    results = []

    for row in iterator:
        ticker = row.Ticker
        name = row.Name

        # Handle suffix
        ticker_sa = ticker if ticker.endswith('.SA') else f"{ticker}.SA"

        stock = yf.Ticker(ticker_sa)

        # Fetch hourly data
        # yfinance returns a Pandas DF, but we iterate it immediately
        # to convert to native Python types for Spark.
        hist = stock.history(start=START_DATE, end=END_DATE, interval="1d")

        if hist.empty:
            continue

        # Iterate over the historical data
        # resetting index makes 'Datetime' a regular column, easier to access
        hist = hist.reset_index()

        for _, h_row in hist.iterrows():
           # Sometime is date, sometimes is datetime
           # Dependens on `interval`
            if 'Date' in h_row:
                dt_val = h_row['Date'].to_pydatetime().date()
            elif 'Datetime' in h_row:
                dt_val = h_row['Datetime'].to_pydatetime().date()
            else:
                continue # Should not happen with standard yfinance

            print(f"{ticker}")

            results.append(Row(
                Ticker=ticker,
                name=name,
                Date=dt_val,
                Open=float(h_row['Open']),
                High=float(h_row['High']),
                Low=float(h_row['Low']),
                Close=float(h_row['Close']),
                Volume=int(h_row['Volume'])
            ))

    return iter(results)

In [ ]:
# Repartition ensures distribution across workers
# If input_df is small, Spark might put it all on 1 partition.
# repartition(10) forces it to spread out.
df.limit(2).show()
rdd_input = df.limit(2).repartition(10).rdd

# 5. Apply mapPartitions
# We convert the DataFrame to RDD, apply the function, and convert back to DataFrame
rdd_output = rdd_input.mapPartitions(process_partition_hourly)

# Apply schema to create the final DataFrame
final_df = spark.createDataFrame(rdd_output, schema=result_schema2)

print("Extraction Results:")
final_df.show()

+-----+------+
| Name|Ticker|
+-----+------+
|   B3| B3SA3|
|Cogna| COGN3|
+-----+------+

Extraction Results:
+------+-----+----------+------------------+------------------+------------------+------------------+--------+
|Ticker| name|  Datetime|              Open|              High|               Low|             Close|  Volume|
+------+-----+----------+------------------+------------------+------------------+------------------+--------+
| COGN3|Cogna|2025-12-12| 3.272688507922782|3.3867492431131154| 3.272688507922782| 3.334105968475342|18911420|
| COGN3|Cogna|2025-12-15| 3.377976149814705|  3.39552337081159| 3.334105911312034| 3.351654052734375|18782940|
| COGN3|Cogna|2025-12-16|3.3253317526634265|3.3253317526634265|3.1849489212036133|3.1849489212036133|38967170|
| B3SA3|   B3|2025-12-12|13.976381407403863| 14.25843778230082|13.840215909071977|13.937477111816406|30560500|
| B3SA3|   B3|2025-12-15|14.141724682586513| 14.20980696695146| 14.01528549194336| 14.01528549194336|38100400|
|

In [ ]:
# --- Usage Example ---

# 1. Define the date range
start = "2025-12-12"
end = "2025-12-17"

# Using a small slice for testing speed
df_history = get_historical_stock_data(df.limit(3).toPandas(), start_date=start, end_date=end)

print(df_history)

Fetching data from 2025-12-12 to 2025-12-17...
[1/3] B3SA3: R$14.66 | Var: 2.02%
[1/3] B3SA3: R$14.47 | Var: -1.30%
[1/3] B3SA3: R$14.35 | Var: -0.83%
[1/3] B3SA3: R$14.39 | Var: 0.28%
[1/3] B3SA3: R$14.41 | Var: 0.14%
[1/3] B3SA3: R$14.40 | Var: -0.07%
[1/3] B3SA3: R$14.38 | Var: -0.21%
[1/3] B3SA3: R$14.46 | Var: -0.55%
[1/3] B3SA3: R$14.51 | Var: 0.35%
[1/3] B3SA3: R$14.55 | Var: 0.28%
[1/3] B3SA3: R$14.56 | Var: 0.14%
[1/3] B3SA3: R$14.49 | Var: -0.41%
[1/3] B3SA3: R$14.48 | Var: 0.00%
[1/3] B3SA3: R$14.47 | Var: 0.00%
[1/3] B3SA3: R$14.06 | Var: -1.68%
[1/3] B3SA3: R$13.99 | Var: -0.50%
[1/3] B3SA3: R$13.90 | Var: -0.64%
[1/3] B3SA3: R$13.93 | Var: 0.22%
[1/3] B3SA3: R$13.89 | Var: -0.29%
[1/3] B3SA3: R$13.85 | Var: -0.36%
[1/3] B3SA3: R$13.75 | Var: -0.79%
[2/3] COGN3: R$3.83 | Var: 2.68%
[2/3] COGN3: R$3.79 | Var: -1.04%
[2/3] COGN3: R$3.75 | Var: -1.06%
[2/3] COGN3: R$3.78 | Var: 0.80%
[2/3] COGN3: R$3.82 | Var: 1.06%
[2/3] COGN3: R$3.83 | Var: 0.26%
[2/3] COGN3: R$3.81 | Var: 

Agora criando um arquivo em parquet pra jogar no S3, depois eu vejo como fazer pra rodar tudo por lá.

In [ ]:
df_history.to_parquet("history.parquet", compression='snappy')

In [ ]:
import sys
import requests
from bs4 import BeautifulSoup
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType, DateType
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
from datetime import date, timedelta

# Standard imports
import yfinance as yf

# Define mandatory and optional arguments with default values

today = date.today()
yesterday = today - timedelta(days=1)

mandatory_fields = ['JOB_NAME']
optional_args_defaults = ['START_DATE', 'END_DATE']

# Combine all keys for getResolvedOptions
# Note: getResolvedOptions expects the keys to have the double-hyphen prefix if passed at runtime via the CLI/API
if ('--{}'.format('START_DATE') in sys.argv):
    all_args_keys = mandatory_fields + optional_args_defaults
else:
    all_args_keys = mandatory_fields

# Retrieve arguments
args = getResolvedOptions(sys.argv, all_args_keys)

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# Verificar como pegar essas datas direto dos parametros quando enviadas.
DATABASE_NAME = "default"
START_DATE = args.get('START_DATE', yesterday.isoformat())
END_DATE = args.get('END_DATE', today.isoformat())

print(f"Job started. from '{START_DATE}' to '{END_DATE}'")

# Schema de dados
result_schema = StructType([
    StructField("Ticker", StringType(), True),
    StructField("name", StringType(), True),
    StructField("Date", DateType(), True),
    StructField("DateProc", DateType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", LongType(), True)
])

def extract_stock_data():
    url = "https://www.dadosdemercado.com.br/acoes"

    # Use a user-agent to mimic a real browser request
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        # Fetch the webpage content
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Check for connection errors

        # Parse the HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the stock data table
        table = soup.find('table')

        if not table:
            print("Table not found on the page.")
            return

        data = []

        # Iterate over all rows in the table
        # We search for 'tr' tags which represent table rows
        rows = table.find_all('tr')

        for row in rows:
            # Find all cells ('td') in the row
            cols = row.find_all('td')

            # We need at least 2 columns (Ticker is usually col 0, Nome is col 1)
            # This check also skips the header row if it uses 'th' tags
            if len(cols) >= 2:
                ticker = cols[0].get_text(strip=True)
                nome = cols[1].get_text(strip=True)

                # Skip if we accidentally picked up the header row text
                if ticker == "Ticker":
                    continue

                # Add to our list with 'Nome' renamed to 'name'
                data.append({
                    "Ticker": ticker,
                    "Name": nome
                })

        # Create a DataFrame for clean display or export
        df = spark.createDataFrame(data)

        return df

    except Exception as e:
        print(f"An error occurred: {e}")

# Processa as acoes
def process_partition_hourly(iterator):
    results = []

    for row in iterator:
        ticker = row.Ticker
        name = row.Name

        # Handle suffix
        ticker_sa = ticker if ticker.endswith('.SA') else f"{ticker}.SA"

        try:
            stock = yf.Ticker(ticker_sa)

            # Fetch data in pandas dataframe format
            hist = stock.history(start=START_DATE, end=END_DATE, interval='1d')

            if hist.empty:
                continue

            # Iterate over the historical data
            # resetting index makes 'Datetime' a regular column, easier to access
            hist = hist.reset_index()

            for _, h_row in hist.iterrows():
               # Sometime is date, sometimes is datetime
               # Dependens on `interval`
                if 'Date' in h_row:
                    dt_val = h_row['Date'].to_pydatetime().date()
                elif 'Datetime' in h_row:
                    dt_val = h_row['Datetime'].to_pydatetime().date()
                else:
                    continue # Should not happen with standard yfinance

                results.append(Row(
                    Ticker=ticker,
                    Name=name,
                    Date=dt_val,
                    DateProc=dt_val,
                    Open=float(h_row['Open']),
                    High=float(h_row['High']),
                    Low=float(h_row['Low']),
                    Close=float(h_row['Close']),
                    Volume=int(h_row['Volume'])
                ))

        except Exception as e:
            print(f"Error processing {ticker}: {str(e)}")
            # Skip this ticker on error

    return iter(results)

job = Job(glueContext)
job.init(args['JOB_NAME'], args)

df = extract_stock_data()
rdd_input = df.repartition(10).rdd
rdd_output = rdd_input.mapPartitions(process_partition_hourly)
final_df = spark.createDataFrame(rdd_output, schema=result_schema)

# Nao precisa escrever no raw, já da pra mandar tudo direto pro GLUE.
# final_df.write.mode("overwrite").partitionBy("ProcDate").parquet("s3://fiap-rm368041-bucket/raw/")

# This is required to use Glue's catalog writing features
final_dyf = DynamicFrame.fromDF(final_df, glueContext, "final_dyf")

# 5. Write to S3 and Update Data Catalog
# This creates the table 'stock_row' in 'DATABASE_NAME'
sink = glueContext.getSink(
    path="s3://fiap-rm368041-bucket/raw/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["DateProc"], # Add ["Date"] here if you want to partition by date
    enableUpdateCatalog=True,
    transformation_ctx="sink"
)

sink.setCatalogInfo(
    catalogDatabase=DATABASE_NAME,
    catalogTableName="stock_raw"
)

# Seguindo as coisas aqui funcionou.
# https://docs.aws.amazon.com/pt_br/glue/latest/dg/update-from-job.html
sink.setFormat("parquet", useGlueParquetWriter=True)
sink.writeFrame(final_dyf)

print(f"Job finished. Table 'stock_raw' created in database '{DATABASE_NAME}'.")

job.commit()

ATHENA SQL para fazer calculos com datas.

SELECT
    *,
    min(low) OVER (PARTITION BY ticker ORDER BY "date" ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as seven_day_min,
    max(high) OVER (PARTITION BY ticker ORDER BY "date" ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as seven_day_max,
    avg(high) OVER (PARTITION BY ticker ORDER BY "date" ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as seven_day_avg
FROM
    "default"."stock_raw"
WHERE date >= date_add('day', -7, "date") AND TICKER = 'VALE3';

👇 Aqui uma cópia da lambda que dispara o job.

In [ ]:
import json
import boto3
import urllib.parse

# Initialize the Glue client
glue_client = boto3.client('glue')

def lambda_handler(event, context):

    print("Evento recebido: " + json.dumps(event, indent=2))

    # Diz a internet que pode voltar varios arquivos.
    for record in event['Records']:
        bucket_name = record['s3']['bucket']['name']

        # File keys are URL encoded (e.g., spaces become + or %20), so we decode them
        file_key = urllib.parse.unquote_plus(record['s3']['object']['key'])

        print(f"File uploaded: s3://{bucket_name}/{file_key}")

        # Job do glue.
        glue_job_name = 'transformar_dados_raw'

        try:
            response = glue_client.start_job_run(
                JobName=glue_job_name,
                Arguments={} # Não to passando nada, mas da pra mandar coisas.
            )
            print(f"Job iniciado. JobRunId: {response['JobRunId']}")

        except Exception as e:
            print(f"Erro: {e}")
            raise e

    return {
        'statusCode': 200,
        'body': json.dumps('Job disparado')
    }

In [ ]:
import json
import boto3
import urllib.parse

glue_client = boto3.client('glue')

def lambda_handler(event, context):
    print("Evento: " + json.dumps(event, indent=2))

    glue_job_name = 'transformar_dados_raw'


    try:
        # Pega as ultimas 5 execuções
        response = glue_client.get_job_runs(
            JobName=glue_job_name,
            MaxResults=5
        )

        # Se for um desses
        active_states = ['STARTING', 'RUNNING', 'STOPPING']

        for run in response.get('JobRuns', []):
            if run['JobRunState'] in active_states:
                print(f"Skipping trigger: Job '{glue_job_name}' is already active. "
                      f"RunID: {run['Id']} | State: {run['JobRunState']}")
                return {
                    'statusCode': 200,
                    'body': json.dumps('Job is already running. Trigger skipped.')
                }

    except Exception as e:
        print(f"Error checking job status: {e}")
        raise e

    for record in event['Records']:
        bucket_name = record['s3']['bucket']['name']
        file_key = urllib.parse.unquote_plus(record['s3']['object']['key'])

        print(f"File uploaded: s3://{bucket_name}/{file_key}")

        try:
            response = glue_client.start_job_run(
                JobName=glue_job_name,
                Arguments={}
            )
            print(f"Transformacao disparada. JobRunId: {response['JobRunId']}")

        except Exception as e:
            print(f"Erro: {e}")
            raise e

    return {
        'statusCode': 200,
        'body': json.dumps('FIM')
    }